# Phase 4: build the shot-level dataset

A raw match event file contains many event types. Here we select the individual shot events, make a binary goal label, calculate shot distance and angle from the documented StatsBomb coordinate frame, and save auditable CSV files. This phase does not encode categories, scale features, split matches, train models, or make charts.

The source is the pinned StatsBomb Open Data La Liga 2015/2016 season (competition 11, season 27). The build script first checks the complete Phase 3 manifest and file hashes. It uses no network. If raw data are missing or changed, repair Phase 3 before continuing.

The main table has one row per eligible shot in periods 1 and 2. Its label is `is_goal`; a recognized non-goal outcome is 0. Own-goal events are audited separately because they are not ordinary shots. The reconciliation report compares shot goals plus the beneficiary's opponent own-goals against each official match score, counting own goals only once.

Read `docs/shot_data_dictionary.md` for field definitions and `docs/model_columns.json` for the future-model feature boundary. In particular, StatsBomb xG is only a reference value. Event and match identifiers, scores, location coordinates, original outcome text, and other audit fields are not model inputs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

starting_directory = Path.cwd().resolve()
search_locations = (starting_directory, *starting_directory.parents)
project_root = next(
    (
        folder for folder in search_locations
        if (folder / ".venv").is_dir()
        and (folder / "README.md").is_file()
        and (folder / "scripts" / "build_shot_dataset.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project folder in VS Code and select its .venv kernel before running this notebook.")
expected_python = (project_root / ".venv").resolve()
if Path(sys.prefix).resolve() != expected_python:
    raise RuntimeError(f"Select the project .venv kernel. Expected {expected_python}; got {Path(sys.prefix).resolve()}")

script_path = project_root / "scripts" / "build_shot_dataset.py"
processed_dir = project_root / "data" / "processed"
print(f"Project folder: {project_root}")
print(f"Python interpreter: {sys.executable}")
print(f"Build script: {script_path}")

## Build or rebuild the derived files

This command reads the pinned local raw files and validates them against the Phase 3 manifest. It writes the derived outputs only after its checks succeed. Re-running it is deterministic for the same source snapshot; CSVs do not contain a build timestamp. If a previous attempt failed, read the error rather than editing source JSON to make it pass.

In [ ]:
build = subprocess.run(
    [sys.executable, str(script_path), "--root", str(project_root)],
    check=True,
    capture_output=True,
    text=True,
)
print(build.stdout)
if build.stderr:
    print(build.stderr)

## Inspect the generated tables

Load identifiers as text where appropriate so a CSV reader does not reinterpret them. The report provides actual counts and hashes for this run. Check the target balance and missingness before proceeding. A row count here describes shots, not events or matches.

In [ ]:
import pandas as pd
from IPython.display import display

shots_path = processed_dir / "shots.csv"
reconciliation_path = processed_dir / "goal_reconciliation.csv"
own_goals_path = processed_dir / "own_goals.csv"
report_path = project_root / "docs" / "shot_dataset_report.json"

shots = pd.read_csv(shots_path, dtype={
    "event_id": "string",
    "match_date": "string",
    "home_team_id": "string",
    "away_team_id": "string",
    "team_id": "string",
    "opponent_id": "string",
    "player_id": "string",
})
reconciliation = pd.read_csv(reconciliation_path)
own_goals = pd.read_csv(own_goals_path, dtype={"event_id": "string"})
report = json.loads(report_path.read_text(encoding="utf-8"))

print(f"Shot rows: {len(shots):,}")
print(f"Match reconciliations: {len(reconciliation):,}")
print(f"Own-goal audit rows: {len(own_goals):,}")
print("Target counts:")
display(shots["is_goal"].value_counts(dropna=False).sort_index().rename_axis("is_goal").to_frame("shots"))
print("Missing values by shot-table column:")
display(shots.isna().sum().rename("missing_values").to_frame())
print("Shot outcome counts:")
display(shots["shot_outcome"].value_counts(dropna=False).rename_axis("shot_outcome").to_frame("shots"))
quality_summary = {
    "status": report["status"],
    "source_commit": report["source"]["commit"],
    "eligible_shots": report["eligible_shot_count"],
    "goal_shots": report["goal_count"],
    "non_goal_shots": report["non_goal_count"],
    "penalty_shots": report["penalty_shot_count"],
    "penalty_goals": report["penalty_goal_count"],
    "own_goal_against_events": report["own_goal_against_records"],
    "matches_reconciled": f"{int(reconciliation['reconciled'].eq(True).sum())}/{len(reconciliation)}",
    "excluded_period_shots": report["excluded_period_shot_count"],
    "duplicate_event_ids": report["duplicate_event_id_count"],
    "shots_missing_freeze_frame": report["freeze_frame_missing_count"],
}
print("Compact quality summary:")
display(pd.Series(quality_summary, name="value").to_frame())
print("Off-camera source flag counts (absent is shown separately):")
display(pd.Series(report["off_camera_counts"], name="shots").rename_axis("event.off_camera").to_frame())
print("Absent context flags (stored as 0 / NOT FLAGGED):")
display(pd.Series(report["raw_flag_absent_counts"], name="shots").rename_axis("context_flag").to_frame())
print("Freeze-frame availability:")
display(pd.Series({
    "present": len(shots) - report["freeze_frame_missing_count"],
    "missing": report["freeze_frame_missing_count"],
}, name="shots").rename_axis("freeze_frame").to_frame())

## Check the model boundary and reconciliation

The feature allowlist is intentionally small and contains only pre-shot or shot-context information selected for the first model comparison. The checks below ensure the stored outcome and provider xG stay outside that list. Match IDs remain available for match-grouped splitting later.

The reconciliation uses `shot goals + opponent Own Goal Against` for each side. `Own Goal For` is retained as corroboration; do not add both representations because that would double-count own goals. If any match does not reconcile, stop and inspect the report and source events.

In [ ]:
model_columns = json.loads((project_root / "docs" / "model_columns.json").read_text(encoding="utf-8"))
feature_names = [item["column"] for item in model_columns["features"]]
forbidden = {"is_goal", "shot_outcome", "statsbomb_xg", "match_id", "event_id", "x", "y", "off_camera"}
assert not forbidden.intersection(feature_names), "A label, identifier, reference, or audit coordinate leaked into the feature allowlist."
assert set(feature_names).issubset(shots.columns), "The feature specification refers to a missing shot-table column."
assert shots["event_id"].is_unique, "Shot event IDs must be unique in this season table."
assert shots["is_goal"].isin([0, 1]).all(), "The target must be binary."
assert reconciliation["reconciled"].eq(True).all(), "At least one official score did not reconcile."
assert set(shots["period"].unique()).issubset({1, 2}), "Only regulation periods 1 and 2 belong in this table."

print(f"Allowed model features ({len(feature_names)}): {feature_names}")
print("Goal reconciliation: all match rows pass.")
print("Shot table preview:")
display(shots.head(8))
print("Reconciliation preview:")
display(reconciliation.head(8))
print("Own-goal audit preview:")
display(own_goals.head(8))

## What to record before moving on

Save this notebook after running it so the tables and actual outputs stay with the project. Confirm that the script reports 380 reconciled matches, review the shot and own-goal counts, and read the generated `docs/shot_dataset_report.json`. The report distinguishes exclusions from invalid data: unexpected periods are excluded and counted; malformed, unrecognized, or incomplete required shot records stop the build. Phase 5 can begin only after the outputs and source hashes are reviewed.